# scispaCy + Regex NER Evaluation on MACCROBAT

This notebook evaluates the shared `models/scispacy_and_regex/scispacy_regex_ner.py` pipeline on the MACCROBAT dataset.

The scispaCy + regex pipeline predicts broad meaning groups, while MACCROBAT uses a richer token-level BIO schema. For a fair scoped benchmark, this notebook maps only compatible MACCROBAT labels into the same meaning groups, filters unsupported prediction groups, and reports exact character-span metrics.

## Imports and Configuration

In [12]:
from __future__ import annotations

from collections import defaultdict
from pathlib import Path
import json
import sys

import pandas as pd
from datasets import load_dataset
from tqdm.auto import tqdm

PROJECT_ROOT = Path.cwd().resolve()
while PROJECT_ROOT != PROJECT_ROOT.parent:
    module_path = PROJECT_ROOT / "models" / "scispacy_and_regex" / "scispacy_regex_ner.py"
    if module_path.exists():
        break
    PROJECT_ROOT = PROJECT_ROOT.parent
else:
    raise FileNotFoundError("Could not locate the Deep-learning-Lab project root.")

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from models.scispacy_and_regex.scispacy_regex_ner import (
    load_available_scispacy_models,
    resolve_overlaps,
    run_regex_patterns,
    run_scispacy_models,
)

DATASET_NAME = "ktgiahieu/maccrobat2018_2020"
SPLIT = "train"
MAX_DOCUMENTS = None  # Set to a small integer such as 5 for a quick smoke test.
OUTPUT_DIR = PROJECT_ROOT / "outputs" / "scispacy-regex-maccrobat-evaluation"

## Load MACCROBAT and scispaCy Models

The five scispaCy models are loaded once and reused for every document. If a model is missing, the loader prints the exact install command. Running all 400 MACCROBAT documents through all available models can take a while on CPU.

In [13]:
maccrobat = load_dataset(DATASET_NAME, split=SPLIT)
evaluation_dataset = (
    maccrobat
    if MAX_DOCUMENTS is None
    else maccrobat.select(range(min(MAX_DOCUMENTS, len(maccrobat))))
)

loaded_models = load_available_scispacy_models()
if not loaded_models:
    raise RuntimeError("No scispaCy models are installed.")

print("Evaluation documents:", len(evaluation_dataset))
print("Loaded models:", list(loaded_models))

Evaluation documents: 400
Loaded models: ['en_core_sci_sm', 'en_ner_bc5cdr_md', 'en_ner_bionlp13cg_md', 'en_ner_craft_md', 'en_ner_jnlpba_md']


## Expanded MACCROBAT Schema Mapping

MACCROBAT has many more labels than the scispaCy + regex pipeline predicts directly. This table documents every MACCROBAT label and maps it to the closest available pipeline meaning group where possible.

By default, the benchmark scores only `direct` mappings. Broad mappings are kept in the table for transparency, but they are excluded from the default score because they pull in very noisy generic spans such as `BIOMEDICAL_MENTION`. To run an experimental broader benchmark, add `"broad"` to `SCORED_MAPPING_STRENGTHS`.


In [14]:
MACCROBAT_LABEL_MAPPINGS = [
    {"maccrobat_label": "Activity", "meaning_group": "BIOMEDICAL_MENTION", "mapping_strength": "broad"},
    {"maccrobat_label": "Administration", "meaning_group": "MEDICATION_ROUTE", "mapping_strength": "broad"},
    {"maccrobat_label": "Age", "meaning_group": "AGE", "mapping_strength": "direct"},
    {"maccrobat_label": "Area", "meaning_group": "MEASUREMENT", "mapping_strength": "broad"},
    {"maccrobat_label": "Biological_attribute", "meaning_group": "BIOMEDICAL_MENTION", "mapping_strength": "broad"},
    {"maccrobat_label": "Biological_structure", "meaning_group": "ANATOMY", "mapping_strength": "direct"},
    {"maccrobat_label": "Clinical_event", "meaning_group": "BIOMEDICAL_MENTION", "mapping_strength": "broad"},
    {"maccrobat_label": "Color", "meaning_group": "BIOMEDICAL_MENTION", "mapping_strength": "broad"},
    {"maccrobat_label": "Coreference", "meaning_group": None, "mapping_strength": "unsupported"},
    {"maccrobat_label": "Date", "meaning_group": "DATE", "mapping_strength": "direct"},
    {"maccrobat_label": "Detailed_description", "meaning_group": "BIOMEDICAL_MENTION", "mapping_strength": "broad"},
    {"maccrobat_label": "Diagnostic_procedure", "meaning_group": "BIOMEDICAL_MENTION", "mapping_strength": "broad"},
    {"maccrobat_label": "Disease_disorder", "meaning_group": "CLINICAL_CONDITION", "mapping_strength": "direct"},
    {"maccrobat_label": "Distance", "meaning_group": "MEASUREMENT", "mapping_strength": "broad"},
    {"maccrobat_label": "Dosage", "meaning_group": "DOSAGE_OR_MEASUREMENT", "mapping_strength": "direct"},
    {"maccrobat_label": "Duration", "meaning_group": "MEDICATION_DURATION", "mapping_strength": "direct"},
    {"maccrobat_label": "Family_history", "meaning_group": "CLINICAL_CONDITION", "mapping_strength": "broad"},
    {"maccrobat_label": "Frequency", "meaning_group": "MEDICATION_FREQUENCY", "mapping_strength": "direct"},
    {"maccrobat_label": "Height", "meaning_group": "MEASUREMENT", "mapping_strength": "broad"},
    {"maccrobat_label": "History", "meaning_group": "BIOMEDICAL_MENTION", "mapping_strength": "broad"},
    {"maccrobat_label": "Lab_value", "meaning_group": "LAB_VALUE", "mapping_strength": "direct"},
    {"maccrobat_label": "Mass", "meaning_group": "MEASUREMENT", "mapping_strength": "broad"},
    {"maccrobat_label": "Medication", "meaning_group": "CHEMICAL_OR_MEDICATION", "mapping_strength": "direct"},
    {"maccrobat_label": "Nonbiological_location", "meaning_group": "CARE_SITE_OR_LOCATION", "mapping_strength": "direct"},
    {"maccrobat_label": "Occupation", "meaning_group": None, "mapping_strength": "unsupported"},
    {"maccrobat_label": "Other_entity", "meaning_group": "BIOMEDICAL_MENTION", "mapping_strength": "broad"},
    {"maccrobat_label": "Other_event", "meaning_group": "BIOMEDICAL_MENTION", "mapping_strength": "broad"},
    {"maccrobat_label": "Outcome", "meaning_group": "CLINICAL_CONDITION", "mapping_strength": "broad"},
    {"maccrobat_label": "Personal_background", "meaning_group": None, "mapping_strength": "unsupported"},
    {"maccrobat_label": "Qualitative_concept", "meaning_group": "BIOMEDICAL_MENTION", "mapping_strength": "broad"},
    {"maccrobat_label": "Quantitative_concept", "meaning_group": "MEASUREMENT", "mapping_strength": "broad"},
    {"maccrobat_label": "Severity", "meaning_group": "BIOMEDICAL_MENTION", "mapping_strength": "broad"},
    {"maccrobat_label": "Sex", "meaning_group": None, "mapping_strength": "unsupported"},
    {"maccrobat_label": "Shape", "meaning_group": "BIOMEDICAL_MENTION", "mapping_strength": "broad"},
    {"maccrobat_label": "Sign_symptom", "meaning_group": "CLINICAL_CONDITION", "mapping_strength": "direct"},
    {"maccrobat_label": "Subject", "meaning_group": None, "mapping_strength": "unsupported"},
    {"maccrobat_label": "Texture", "meaning_group": "BIOMEDICAL_MENTION", "mapping_strength": "broad"},
    {"maccrobat_label": "Therapeutic_procedure", "meaning_group": "BIOMEDICAL_MENTION", "mapping_strength": "broad"},
    {"maccrobat_label": "Time", "meaning_group": "TIME", "mapping_strength": "direct"},
    {"maccrobat_label": "Volume", "meaning_group": "MEASUREMENT", "mapping_strength": "broad"},
    {"maccrobat_label": "Weight", "meaning_group": "MEASUREMENT", "mapping_strength": "broad"},
]

SCORED_MAPPING_STRENGTHS = {"direct"}  # Add "broad" to include approximate mappings.

mapping_table = pd.DataFrame(MACCROBAT_LABEL_MAPPINGS)
MACCROBAT_TO_MEANING_GROUP = {
    row["maccrobat_label"]: row["meaning_group"]
    for row in MACCROBAT_LABEL_MAPPINGS
    if row["meaning_group"] is not None
    and row["mapping_strength"] in SCORED_MAPPING_STRENGTHS
}
SCOPED_GROUPS = set(MACCROBAT_TO_MEANING_GROUP.values())
UNSUPPORTED_MACCROBAT_LABELS = sorted(
    row["maccrobat_label"]
    for row in MACCROBAT_LABEL_MAPPINGS
    if row["meaning_group"] is None
)
IGNORED_UNSCORED_MACCROBAT_LABELS = sorted(
    row["maccrobat_label"]
    for row in MACCROBAT_LABEL_MAPPINGS
    if row["maccrobat_label"] not in MACCROBAT_TO_MEANING_GROUP
)

print("Scored mapping strengths:", sorted(SCORED_MAPPING_STRENGTHS))
print("Scored MACCROBAT labels:", len(MACCROBAT_TO_MEANING_GROUP))
print("Ignored/unscored MACCROBAT labels:", len(IGNORED_UNSCORED_MACCROBAT_LABELS))

mapping_table


Scored mapping strengths: ['direct']
Scored MACCROBAT labels: 12
Ignored/unscored MACCROBAT labels: 29


,maccrobat_label,meaning_group,mapping_strength
0,Activity,BIOMEDICAL_MENTION,broad
1,Administration,MEDICATION_ROUTE,broad
2,Age,AGE,direct
3,Area,MEASUREMENT,broad
4,Biological_attribute,BIOMEDICAL_MENTION,broad
5,Biological_structure,ANATOMY,direct
6,Clinical_event,BIOMEDICAL_MENTION,broad
7,Color,BIOMEDICAL_MENTION,broad
8,Coreference,NaN,unsupported
9,Date,DATE,direct


## Gold Span Helpers

MACCROBAT provides token lists and BIO tags. This reconstructs each document with single spaces between tokens and converts compatible BIO spans into document-level character offsets.

In [15]:
def strip_bio_prefix(tag: str) -> str:
    return tag[2:] if tag.startswith(("B-", "I-")) else tag


def build_document(tokens: list[str]) -> tuple[str, list[int], list[int]]:
    starts = []
    ends = []
    cursor = 0

    for index, token in enumerate(tokens):
        if index:
            cursor += 1
        starts.append(cursor)
        cursor += len(token)
        ends.append(cursor)

    return " ".join(tokens), starts, ends


def build_gold_entities(tags: list[str], starts: list[int], ends: list[int]) -> list[tuple[str, int, int]]:
    entities = []
    current_group = None
    current_start = None
    current_end = None

    def close_current() -> None:
        nonlocal current_group, current_start, current_end
        if current_group is not None and current_start is not None and current_end is not None:
            entities.append((current_group, current_start, current_end))
        current_group = None
        current_start = None
        current_end = None

    for tag, token_start, token_end in zip(tags, starts, ends):
        raw_label = strip_bio_prefix(tag)
        group = MACCROBAT_TO_MEANING_GROUP.get(raw_label)

        if tag == "O" or group is None:
            close_current()
            continue

        starts_new = tag.startswith("B-") or group != current_group
        if starts_new:
            close_current()
            current_group = group
            current_start = token_start

        current_end = token_end

    close_current()
    return entities

## Run One Document

This uses the same functions as the command-line pipeline: scispaCy candidates, regex candidates, then overlap resolution. Predictions outside the compatible MACCROBAT groups are filtered before scoring.

In [16]:
def run_pipeline_on_document(tokens: list[str]):
    text, _, _ = build_document(tokens)
    next_candidate_id = 1

    scispacy_candidates, next_candidate_id = run_scispacy_models(
        text,
        loaded_models,
        next_candidate_id,
    )
    regex_candidates, _ = run_regex_patterns(text, next_candidate_id)
    selected = resolve_overlaps(scispacy_candidates + regex_candidates)

    return [
        item.candidate
        for item in selected
        if item.candidate.meaning_group in SCOPED_GROUPS
        and item.candidate.start >= 0
        and item.candidate.end > item.candidate.start
    ]


sample_text, sample_starts, sample_ends = build_document(evaluation_dataset[0]["tokens"])
sample_gold = build_gold_entities(evaluation_dataset[0]["tags"], sample_starts, sample_ends)
sample_predictions = run_pipeline_on_document(evaluation_dataset[0]["tokens"])

pd.DataFrame(
    [
        {
            "meaning_group": candidate.meaning_group,
            "text": candidate.text,
            "start": candidate.start,
            "end": candidate.end,
            "method": candidate.method,
            "source": candidate.source,
            "label": candidate.label,
        }
        for candidate in sample_predictions[:25]
    ]
)

,meaning_group,text,start,end,method,source,label
0,CLINICAL_CONDITION,jaundice,123,131,scispacy,en_ner_bc5cdr_md,DISEASE
1,DOSAGE_OR_MEASUREMENT,9.9 kg,174,180,regex,regex:DOSAGE,DOSAGE
2,DOSAGE_OR_MEASUREMENT,6.84 mg,482,489,regex,regex:DOSAGE,DOSAGE
3,DOSAGE_OR_MEASUREMENT,9.18 mg,519,526,regex,regex:DOSAGE,DOSAGE
4,DOSAGE_OR_MEASUREMENT,1880 mg,1061,1068,regex,regex:DOSAGE,DOSAGE
5,DOSAGE_OR_MEASUREMENT,640 mg,1085,1091,regex,regex:DOSAGE,DOSAGE
6,DOSAGE_OR_MEASUREMENT,890 g,1179,1184,regex,regex:DOSAGE,DOSAGE
7,DOSAGE_OR_MEASUREMENT,3 g,1199,1202,regex,regex:DOSAGE,DOSAGE
8,ANATOMY,liver,1316,1321,scispacy,en_ner_bionlp13cg_md,ORGAN
9,ANATOMY,hilum,1420,1425,scispacy,en_ner_bionlp13cg_md,MULTI_TISSUE_STRUCTURE


## Evaluate Entity Spans

A prediction counts as correct when it has the same meaning group as a MACCROBAT gold entity, the spans overlap, and one normalized text is a subset of the other. For example, `severe knee pain` and `knee pain` count as a match.


In [17]:
def spans_overlap(left: dict, right: dict) -> bool:
    return left["start"] < right["end"] and right["start"] < left["end"]


def normalized_entity_text(text: str) -> str:
    return " ".join(str(text).casefold().split())


def is_entity_match(gold_entity: dict, predicted_entity: dict) -> bool:
    if gold_entity["meaning_group"] != predicted_entity["meaning_group"]:
        return False
    if not spans_overlap(gold_entity, predicted_entity):
        return False

    gold_text = normalized_entity_text(gold_entity["text"])
    predicted_text = normalized_entity_text(predicted_entity["text"])
    if not gold_text or not predicted_text:
        return False

    return gold_text in predicted_text or predicted_text in gold_text


def score_entities(gold_entities: list[dict], predicted_entities: list[dict]) -> dict:
    matched_prediction_indexes = set()
    matched_gold_indexes = set()

    for gold_index, gold_entity in enumerate(gold_entities):
        best_prediction_index = None
        best_prediction_length = None

        for prediction_index, predicted_entity in enumerate(predicted_entities):
            if prediction_index in matched_prediction_indexes:
                continue
            if not is_entity_match(gold_entity, predicted_entity):
                continue

            prediction_length = predicted_entity["end"] - predicted_entity["start"]
            if best_prediction_length is None or prediction_length < best_prediction_length:
                best_prediction_index = prediction_index
                best_prediction_length = prediction_length

        if best_prediction_index is not None:
            matched_gold_indexes.add(gold_index)
            matched_prediction_indexes.add(best_prediction_index)

    return {
        "tp": len(matched_gold_indexes),
        "fp": len(predicted_entities) - len(matched_prediction_indexes),
        "fn": len(gold_entities) - len(matched_gold_indexes),
    }


def update_group_counts(group_counts, gold_entities: list[dict], predicted_entities: list[dict]) -> None:
    for group in sorted(SCOPED_GROUPS):
        group_gold = [entity for entity in gold_entities if entity["meaning_group"] == group]
        group_predicted = [entity for entity in predicted_entities if entity["meaning_group"] == group]
        scores = score_entities(group_gold, group_predicted)
        for key, value in scores.items():
            group_counts[group][key] += value


group_counts = defaultdict(lambda: {"tp": 0, "fp": 0, "fn": 0})
document_rows = []

for document_index, row in enumerate(tqdm(evaluation_dataset, desc="Evaluating scispaCy + regex")):
    text, starts, ends = build_document(row["tokens"])
    gold_entities = [
        {
            "meaning_group": group,
            "start": start,
            "end": end,
            "text": text[start:end],
        }
        for group, start, end in build_gold_entities(row["tags"], starts, ends)
    ]
    predicted_entities = [
        {
            "meaning_group": candidate.meaning_group,
            "start": candidate.start,
            "end": candidate.end,
            "text": candidate.text,
        }
        for candidate in run_pipeline_on_document(row["tokens"])
    ]

    scores = score_entities(gold_entities, predicted_entities)
    update_group_counts(group_counts, gold_entities, predicted_entities)

    document_rows.append(
        {
            "document": document_index,
            "gold_entities": len(gold_entities),
            "predicted_entities": len(predicted_entities),
            "true_positives": scores["tp"],
            "false_positives": scores["fp"],
            "false_negatives": scores["fn"],
        }
    )

document_results = pd.DataFrame(document_rows)
document_results.head()


Evaluating scispaCy + regex:   0%|          | 0/400 [00:00<?, ?it/s]

,document,gold_entities,predicted_entities,true_positives,false_positives,false_negatives
0,0,45,11,4,7,41
1,1,35,30,2,28,33
2,2,41,7,3,4,38
3,3,17,19,4,15,13
4,4,31,11,0,11,31


## Results and Saved Artifacts

The notebook writes the standard evaluation artifacts:

- `summary.json`
- `document_results.csv`
- `per_group_results.csv`


In [18]:
def safe_divide(numerator: int, denominator: int) -> float:
    return numerator / denominator if denominator else 0.0


result_rows = []
for group in sorted(SCOPED_GROUPS):
    counts = group_counts[group]
    precision = safe_divide(counts["tp"], counts["tp"] + counts["fp"])
    recall = safe_divide(counts["tp"], counts["tp"] + counts["fn"])
    f1 = safe_divide(2 * precision * recall, precision + recall)
    result_rows.append(
        {
            "meaning_group": group,
            **counts,
            "precision": precision,
            "recall": recall,
            "f1": f1,
        }
    )

per_group_results = pd.DataFrame(result_rows)

total_tp = int(per_group_results["tp"].sum())
total_fp = int(per_group_results["fp"].sum())
total_fn = int(per_group_results["fn"].sum())
micro_precision = safe_divide(total_tp, total_tp + total_fp)
micro_recall = safe_divide(total_tp, total_tp + total_fn)
micro_f1 = safe_divide(2 * micro_precision * micro_recall, micro_precision + micro_recall)

summary = {
    "dataset": DATASET_NAME,
    "split": SPLIT,
    "documents": len(evaluation_dataset),
    "max_documents": MAX_DOCUMENTS,
    "loaded_models": list(loaded_models),
    "matching": "same meaning_group + overlapping spans + one normalized text contains the other",
    "scored_mapping_strengths": sorted(SCORED_MAPPING_STRENGTHS),
    "scored_maccrobat_labels": sorted(MACCROBAT_TO_MEANING_GROUP),
    "ignored_unscored_maccrobat_labels": IGNORED_UNSCORED_MACCROBAT_LABELS,
    "ignored_unscored_label_count": len(IGNORED_UNSCORED_MACCROBAT_LABELS),
    "unsupported_maccrobat_labels": UNSUPPORTED_MACCROBAT_LABELS,
    "true_positives": total_tp,
    "false_positives": total_fp,
    "false_negatives": total_fn,
    "precision": micro_precision,
    "recall": micro_recall,
    "f1": micro_f1,
}

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
document_results.to_csv(OUTPUT_DIR / "document_results.csv", index=False)
per_group_results.to_csv(OUTPUT_DIR / "per_group_results.csv", index=False)
with (OUTPUT_DIR / "summary.json").open("w", encoding="utf-8") as output_file:
    json.dump(summary, output_file, indent=2)

display(pd.Series(summary, name="micro_metrics"))
per_group_results.sort_values("f1", ascending=False)


dataset                                                   ktgiahieu/maccrobat2018_2020
split                                                                            train
documents                                                                          400
max_documents                                                                     None
loaded_models                        [en_core_sci_sm, en_ner_bc5cdr_md, en_ner_bion...
matching                             same meaning_group + overlapping spans + one n...
scored_mapping_strengths                                                      [direct]
scored_maccrobat_labels              [Age, Biological_structure, Date, Disease_diso...
ignored_unscored_maccrobat_labels    [Activity, Administration, Area, Biological_at...
ignored_unscored_label_count                                                        29
unsupported_maccrobat_labels         [Coreference, Occupation, Personal_background,...
true_positives                             

,meaning_group,tp,fp,fn,precision,recall,f1
8,MEDICATION_DURATION,124,42,285,0.746988,0.303178,0.431304
6,DOSAGE_OR_MEASUREMENT,243,895,115,0.213533,0.678771,0.324866
1,ANATOMY,913,2983,1338,0.234343,0.405598,0.297055
9,MEDICATION_FREQUENCY,24,190,44,0.112150,0.352941,0.170213
3,CHEMICAL_OR_MEDICATION,162,710,1284,0.185780,0.112033,0.139776
0,AGE,24,42,376,0.363636,0.060000,0.103004
2,CARE_SITE_OR_LOCATION,28,26,477,0.518519,0.055446,0.100179
4,CLINICAL_CONDITION,195,491,6069,0.284257,0.031130,0.056115
5,DATE,0,0,994,0.000000,0.000000,0.000000
7,LAB_VALUE,0,60,2534,0.000000,0.000000,0.000000
